# 01 — Yahoo Finance dataset builder (paper §5.3) — v2

Builds the supervised datasets the paper uses for its **baselines** (ANN / RNN / KAN). The PINN itself needs *no* market data.

| Paper (§5.3) | Implementation |
|---|---|
| Tickers `AAPL, MSFT, GOOGL, AMZN, META, F, T, NOK, CCL, SOFI, SNAP, NIO, AAL, PBR, RIVN` | `TICKERS` |
| For each expiry–strike pair: **put chain for American**, **call chain for European** | every expiry of every ticker is downloaded (`MAX_EXPIRIES_PER_TICKER=None`); rows tagged `american_put` / `european_call` |
| Label = mid-quote `(bid+ask)/2`, fall back to last traded price | `price_source = mid / last` |
| No-arbitrage clipping: American put `L = max(K−S,0), U = K`; European call `L = max(S−Ke^{−rτ},0), U = S` | `instrument_bounds`, flag `was_clipped` |
| Static covariates `[log(S/K), τ, σ_implied]` | columns `log_moneyness`, `tau`, `sigma` |
| RNN sequences: per-ticker daily log-returns and realised vol (rolling std × √252), last `W = 60` days | `history_returns.csv` |

**Pipeline (v2).** `download_raw()` (network) → `data/options_raw.csv` (untouched Yahoo rows) → `clean_options()` (pure function, no network, prints a per-filter *funnel*) → `data/options_clean.csv`.
Because the raw snapshot is saved, you can re-run / tweak the cleaning offline without re-downloading — important, since Yahoo quotes change every minute and a given snapshot can never be re-downloaded.
Set `USE_SAVED_RAW = True` to skip the download and re-clean `data/options_raw.csv`.

**What v2 fixes vs. the first notebook**
* one ticker / one expiry / calls only → 15 tickers, all expiries, calls **and** puts;
* filters silently overwritten by a later cell → every filter lives in `clean_options()` and is counted in the funnel;
* quotes below intrinsic value were kept → clipped to the paper's bounds and flagged;
* absurd implied vols (e.g. 453 %) and stale prints (`lastPrice` with zero volume) → filtered;
* `τ` computed from the run-time date → the snapshot date is stored in the data;
* hard-coded `r` → 13-week T-bill (`^IRX`), fallback 4 %;
* network flakiness (Yahoo rate-limits) → retries with back-off, failed tickers are listed, raw data saved;
* duplicates, non-numeric fields, crossed quotes handled; a **support check** shows whether the data covers the four `(K, σ, T)` parameter sets of Tables 1–2.

> **Notes.** `yfinance` talks to Yahoo's API directly — you do **not** need the `uk.finance.yahoo.com` web page (that page is just a view of the same data; scraping it is unnecessary and against Yahoo's terms).
> Yahoo's `impliedVolatility` is computed by Yahoo (own rate/dividend inputs) and is unreliable for illiquid / deep-ITM options; underlying close and option quotes are not time-synchronised; AAPL, MSFT, T, F… pay dividends while the paper's bounds assume none. These are limitations of the paper's setup too.
> **Run it on a trading day, during or just after US market hours** — on weekends/holidays many `bid/ask` are 0 and far fewer rows survive.

In [ ]:
# pip install yfinance pandas numpy matplotlib   (if needed)
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
try:
    import yfinance as yf
except ImportError:
    yf = None
    print("yfinance not installed -> only USE_SAVED_RAW mode will work (pip install yfinance)")
try:
    from IPython.display import display
except Exception:
    display = print

TICKERS = ["AAPL", "MSFT", "GOOGL", "AMZN", "META", "F", "T", "NOK", "CCL", "SOFI", "SNAP", "NIO", "AAL", "PBR", "RIVN"]

SNAPSHOT_DATE = None          # e.g. "2026-10-08"; None -> today
USE_SAVED_RAW = False         # True -> skip download, re-clean data/options_raw.csv
MAX_EXPIRIES_PER_TICKER = None  # None -> every expiry (paper: "each expiry-strike pair"); int -> evenly spaced subset (faster)
MIN_TAU_DAYS, MAX_TAU_YEARS = 3, 2.0
MIN_IV, MAX_IV = 0.02, 2.0    # drop implied vols outside [2 %, 200 %]  (Yahoo returns junk for stale deep-ITM quotes)
MAX_REL_SPREAD = 0.5          # drop quotes with (ask-bid)/mid above this
MAX_ABS_LOG_MONEYNESS = 0.8   # keep |log(S/K)| <= 0.8
HISTORY_PERIOD = "2y"
RV_WINDOW = 20                # rolling window for realised vol (annualised with sqrt(252))
DEFAULT_R = 0.04
SLEEP_BETWEEN_CALLS = 0.3
OUT_DIR = "data"
os.makedirs(OUT_DIR, exist_ok=True)

snapshot = pd.Timestamp(SNAPSHOT_DATE) if SNAPSHOT_DATE else pd.Timestamp.today().normalize()
print("snapshot date:", snapshot.date(), "| weekday:", snapshot.day_name())
if snapshot.dayofweek >= 5:
    print("WARNING: weekend snapshot - Yahoo bid/ask are often 0 outside market hours; expect many fewer usable rows.")

## 1. Pure functions (no network): bounds, labels, cleaning with a funnel report

In [ ]:
RAW_COLS = ["contractSymbol", "strike", "bid", "ask", "lastPrice", "volume", "openInterest", "impliedVolatility"]

def instrument_bounds(instrument, S, K, tau, r):
    '''No-arbitrage bounds exactly as in §5.3 (non-dividend).'''
    S, K, tau = np.asarray(S, float), np.asarray(K, float), np.asarray(tau, float)
    if instrument == "european_call":
        return np.maximum(S - K * np.exp(-r * tau), 0.0), S
    if instrument == "american_put":
        return np.maximum(K - S, 0.0), K * np.ones_like(S)
    raise ValueError(instrument)

def clean_one_instrument(raw, instrument, r, snapshot, funnel):
    '''Clean the raw rows of ONE instrument type. `raw` has RAW_COLS + ticker, expiry, S. Returns the clean frame.'''
    d = raw.copy()
    def log_step(name):
        funnel.append((instrument, name, len(d)))
    log_step("raw rows")
    for c in ("strike", "bid", "ask", "lastPrice", "volume", "openInterest", "impliedVolatility", "S"):
        d[c] = pd.to_numeric(d[c], errors="coerce")
    d["expiry"] = pd.to_datetime(d["expiry"])
    d = d.drop_duplicates(subset=["contractSymbol"]); log_step("after de-duplication")
    d = d[(d["strike"] > 0) & (d["S"] > 0) & np.isfinite(d["strike"])]; log_step("valid strike / spot")

    # ---- label: mid-quote, else last traded price ----
    good_quote = d["bid"].notna() & d["ask"].notna() & (d["bid"] > 0) & (d["ask"] > 0) & (d["ask"] >= d["bid"])
    has_last = d["lastPrice"].notna() & (d["lastPrice"] > 0)
    d["price_source"] = np.where(good_quote, "mid", np.where(has_last, "last", "invalid"))
    d["y_raw"] = np.where(good_quote, 0.5 * (d["bid"] + d["ask"]), np.where(has_last, d["lastPrice"], np.nan))
    d = d[d["price_source"] != "invalid"]; log_step("has a usable price (mid or last)")

    # ---- staleness: a 'last'-priced row with no volume is a stale print, not a market price ----
    stale = (d["price_source"] == "last") & (d["volume"].fillna(0) <= 0)
    d = d[~stale]; log_step("not a stale last-trade print")

    # ---- quote quality ----
    mid = 0.5 * (d["bid"] + d["ask"])
    rel_spread = np.where(d["price_source"] == "mid", (d["ask"] - d["bid"]) / mid, 0.0)
    d = d[rel_spread <= MAX_REL_SPREAD]; log_step(f"relative spread <= {MAX_REL_SPREAD}")
    d = d[np.isfinite(d["impliedVolatility"]) & (d["impliedVolatility"] >= MIN_IV) & (d["impliedVolatility"] <= MAX_IV)]
    log_step(f"implied vol in [{MIN_IV}, {MAX_IV}]")

    # ---- covariates ----
    d["tau"] = np.maximum((d["expiry"] - snapshot).dt.days, 1) / 365.0
    d = d[(d["tau"] * 365 >= MIN_TAU_DAYS) & (d["tau"] <= MAX_TAU_YEARS)]; log_step(f"tau in [{MIN_TAU_DAYS}d, {MAX_TAU_YEARS}y]")
    d["log_moneyness"] = np.log(d["S"] / d["strike"])
    d = d[d["log_moneyness"].abs() <= MAX_ABS_LOG_MONEYNESS]; log_step(f"|log(S/K)| <= {MAX_ABS_LOG_MONEYNESS}")

    # ---- paper's no-arbitrage clipping ----
    L, U = instrument_bounds(instrument, d["S"].values, d["strike"].values, d["tau"].values, r)
    d["L"], d["U"] = L, U
    d = d[d["U"] - d["L"] > 1e-8]; log_step("non-degenerate bounds (U > L)")
    d["y"] = np.clip(d["y_raw"].values, d["L"].values, d["U"].values)
    d["was_clipped"] = ~np.isclose(d["y"], d["y_raw"], atol=1e-9)

    out = pd.DataFrame({
        "ticker": d["ticker"].values, "snapshot_date": snapshot, "instrument": instrument,
        "contractSymbol": d["contractSymbol"].values, "expiry": d["expiry"].values,
        "S": d["S"].values, "K": d["strike"].values, "tau": d["tau"].values, "sigma": d["impliedVolatility"].values, "r": r,
        "bid": d["bid"].values, "ask": d["ask"].values, "lastPrice": d["lastPrice"].values,
        "volume": d["volume"].values, "openInterest": d["openInterest"].values,
        "price_source": d["price_source"].values, "y_raw": d["y_raw"].values,
        "log_moneyness": d["log_moneyness"].values, "L": d["L"].values, "U": d["U"].values,
        "y": d["y"].values, "was_clipped": d["was_clipped"].values,
    })
    return out

def clean_options(raw, r, snapshot):
    '''raw: DataFrame with columns RAW_COLS + [ticker, expiry, S, instrument]. Returns (clean, funnel DataFrame).'''
    funnel, parts = [], []
    for inst in ("european_call", "american_put"):
        sub = raw[raw["instrument"] == inst]
        if len(sub): parts.append(clean_one_instrument(sub, inst, r, snapshot, funnel))
    clean = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    if len(clean):
        clean = clean.sort_values(["ticker", "instrument", "expiry", "K"]).reset_index(drop=True)
        # final invariants the baselines rely on
        assert (clean["y"] >= clean["L"] - 1e-9).all() and (clean["y"] <= clean["U"] + 1e-9).all()
        assert (clean["tau"] > 0).all() and (clean["sigma"] > 0).all() and clean[["S", "K", "y"]].notna().all().all()
    fn = pd.DataFrame(funnel, columns=["instrument", "step", "rows"])
    return clean, fn

def pick_expiries(options, snapshot):
    ok = [e for e in options if MIN_TAU_DAYS <= (pd.Timestamp(e) - snapshot).days <= MAX_TAU_YEARS * 365]
    if MAX_EXPIRIES_PER_TICKER and len(ok) > MAX_EXPIRIES_PER_TICKER:
        ok = [ok[i] for i in np.unique(np.linspace(0, len(ok) - 1, MAX_EXPIRIES_PER_TICKER).round().astype(int))]
    return ok

def support_report(clean, sets=((30., .15, .25), (45., .20, .5), (45., .35, .5), (60., .30, 1.0))):
    '''How many clean rows lie near each (K, sigma, T) parameter set of paper Tables 1-2 (in tau and sigma)?
    The baselines are evaluated at those settings, so empty neighbourhoods mean extrapolation.'''
    rows = []
    for inst in ("european_call", "american_put"):
        d = clean[clean["instrument"] == inst]
        for i, (K, sg, T) in enumerate(sets, 1):
            tol = 0.12 if T <= 0.5 else 0.25
            near_tau = d[(d["tau"] - T).abs() <= tol]
            near_both = near_tau[(near_tau["sigma"] - sg).abs() <= 0.08]
            rows.append(dict(instrument=inst, set=f"Set {i}", K=K, sigma=sg, T=T, rows_near_tau=len(near_tau), rows_near_tau_and_sigma=len(near_both)))
    return pd.DataFrame(rows)

## 2. Risk-free rate, underlying history, realised-vol sequences

In [ ]:
def with_retry(fn, tries=4, base_sleep=1.5):
    for a in range(tries):
        try:
            return fn()
        except Exception as e:
            if a == tries - 1: raise
            time.sleep(base_sleep * (a + 1))

def get_risk_free(default=DEFAULT_R):
    try:
        h = with_retry(lambda: yf.Ticker("^IRX").history(period="5d")["Close"].dropna())
        r = float(h.iloc[-1]) / 100.0
        if 0.0 <= r < 0.25: return r
    except Exception as e:
        print("risk-free fetch failed -> default:", type(e).__name__, e)
    return default

def get_history(ticker, period=HISTORY_PERIOD):
    h = with_retry(lambda: yf.Ticker(ticker).history(period=period)["Close"].dropna())
    h.index = pd.to_datetime(h.index).tz_localize(None).normalize()
    return h

def returns_frame(close, ticker, rv_window=RV_WINDOW):
    lr = np.log(close).diff()
    rv = lr.rolling(rv_window).std() * np.sqrt(252)
    return pd.DataFrame({"ticker": ticker, "date": close.index, "log_ret": lr.values, "rv": rv.values}).dropna()

R = get_risk_free() if (yf is not None and not USE_SAVED_RAW) else DEFAULT_R
print(f"risk-free rate r = {R:.4f}")

## 3. Download raw chains (all tickers × expiries × calls & puts) — or load the saved raw snapshot

In [ ]:
def download_raw():
    raw_frames, hist_frames, spot, failed = [], [], {}, []
    for tk in TICKERS:
        try:
            close = get_history(tk); close = close[close.index <= snapshot]
            S = float(close.iloc[-1]); spot[tk] = S
            hist_frames.append(returns_frame(close, tk))
            t = yf.Ticker(tk); n0 = sum(len(f) for f in raw_frames)
            expiries = pick_expiries(with_retry(lambda: t.options), snapshot)
            for exp in expiries:
                try:
                    oc = with_retry(lambda: t.option_chain(exp))
                except Exception as e:
                    print(f"   {tk} {exp}: option_chain failed ({type(e).__name__})"); continue
                for inst, ch in (("european_call", oc.calls), ("american_put", oc.puts)):
                    if ch is None or len(ch) == 0: continue
                    f = ch.reindex(columns=RAW_COLS).copy()
                    f["ticker"], f["expiry"], f["S"], f["instrument"] = tk, pd.Timestamp(exp), S, inst
                    raw_frames.append(f)
                time.sleep(SLEEP_BETWEEN_CALLS)
            print(f"{tk:6s} S={S:9.2f} expiries={len(expiries):2d} raw rows: {sum(len(f) for f in raw_frames) - n0}")
        except Exception as e:
            print(f"{tk}: FAILED ({type(e).__name__}: {e})"); failed.append(tk)
    raw = pd.concat(raw_frames, ignore_index=True) if raw_frames else pd.DataFrame()
    hist = pd.concat(hist_frames, ignore_index=True) if hist_frames else pd.DataFrame()
    return raw, hist, spot, failed

RAW_F, HIST_F, META_F = f"{OUT_DIR}/options_raw.csv", f"{OUT_DIR}/history_returns.csv", f"{OUT_DIR}/meta.json"
if USE_SAVED_RAW:
    raw = pd.read_csv(RAW_F, parse_dates=["expiry"]); history = pd.read_csv(HIST_F, parse_dates=["date"])
    meta_old = json.load(open(META_F)); snapshot = pd.Timestamp(meta_old["snapshot"]); R = meta_old["r"]; spot = meta_old["spot"]; failed = meta_old.get("failed", [])
    print(f"loaded saved raw snapshot of {snapshot.date()} (r={R:.4f}): {len(raw)} raw rows")
else:
    assert yf is not None, "pip install yfinance"
    raw, history, spot, failed = download_raw()
    raw["snapshot_date"] = snapshot
    raw.to_csv(RAW_F, index=False); history.to_csv(HIST_F, index=False)   # saved BEFORE cleaning
    print(f"\nTotal raw option rows: {len(raw)} | history rows: {len(history)} | failed tickers: {failed or 'none'}")
assert len(raw) > 0, "No option data downloaded - check internet connection / yfinance version (pip install -U yfinance)."

## 4. Clean, save, sanity-check

In [ ]:
options, funnel = clean_options(raw, R, snapshot)
assert len(options) > 0, "All rows were filtered out - is the market closed (bid/ask = 0)? Try a trading day or relax filters."

print("Cleaning funnel (rows surviving each step):")
display(funnel.pivot_table(index="step", columns="instrument", values="rows", sort=False))

options.to_csv(f"{OUT_DIR}/options_clean.csv", index=False)
json.dump(dict(snapshot=str(snapshot.date()), r=R, tickers=TICKERS, spot=spot, failed=failed,
               filters=dict(MIN_TAU_DAYS=MIN_TAU_DAYS, MAX_TAU_YEARS=MAX_TAU_YEARS, MIN_IV=MIN_IV, MAX_IV=MAX_IV,
                            MAX_REL_SPREAD=MAX_REL_SPREAD, MAX_ABS_LOG_MONEYNESS=MAX_ABS_LOG_MONEYNESS)),
          open(META_F, "w"), indent=1)

print("\nClean rows per ticker x instrument:")
display(options.groupby(["ticker", "instrument"]).size().unstack(fill_value=0))
print("Rows per instrument / price source:")
display(options.groupby(["instrument", "price_source"]).size().unstack(fill_value=0))
print("Share of labels clipped to no-arbitrage bounds:")
display(options.groupby("instrument")["was_clipped"].mean().to_frame("clipped_share"))
display(options[["tau", "sigma", "log_moneyness", "y"]].describe().T)

missing = [t for t in TICKERS if t not in set(options["ticker"])]
if missing: print("WARNING - tickers with NO clean rows:", missing)
print("\nSupport of the data around the paper's Table 1-2 parameter sets (K, sigma, T):")
display(support_report(options))
print("(If a row shows 0 in the last column the baselines must extrapolate there -> larger errors than the paper. Re-run on a trading day / relax filters.)")

assert (options["y"] >= options["L"] - 1e-9).all() and (options["y"] <= options["U"] + 1e-9).all()
print("OK: all labels lie within the no-arbitrage bounds.")

fig, ax = plt.subplots(1, 3, figsize=(17, 4))
for inst, a in zip(("european_call", "american_put"), ax[:2]):
    d = options[options["instrument"] == inst]
    a.scatter(d["log_moneyness"], d["sigma"], s=4, alpha=.4); a.set(title=f"{inst}: implied vol smile", xlabel="log(S/K)", ylabel="σ_implied"); a.grid(alpha=.3)
ax[2].hist(options["tau"], bins=40); ax[2].set(title="time to maturity τ (years)", xlabel="τ"); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.show()
print("\nNext: run 02_at_pinn_full_paper.ipynb (it reads data/options_clean.csv and data/history_returns.csv).")